# Reliance Industries (RIL) Stock Price Prediction
### Mini-Project — Semester 5 (2026)

**Goal:** Download real historical data for RELIANCE.NS (NSE), train MLP, LSTM and GRU models on a 60-day sliding window to predict the next day's closing price, and compare all three against a persistence baseline.

**Price definition:** All prices are **adjusted closing prices** as returned by `yfinance` with `auto_adjust=True`. This accounts for corporate actions (splits, dividends) so the price series is comparable across the full history.

**Pipeline**
1. Download and save historical data (2020 – present)
2. Clean and explore the data
3. Chronological train / validation / test split
4. Fit MinMaxScaler on training data only (no leakage)
5. Build sliding-window sequences
6. Train MLP, LSTM, GRU with early stopping
7. Evaluate with MAE and RMSE on the same test period
8. Compare against persistence baseline
9. Save models and scaler for `app.py`

## 1. Imports and Configuration

In [ ]:
import os
import random
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import tensorflow as tf

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout, Flatten, LSTM, GRU
from tensorflow.keras.callbacks import EarlyStopping

warnings.filterwarnings("ignore")

# ── reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ── hyper-parameters ─────────────────────────────────────────────────────────
TICKER      = "RELIANCE.NS"
START_DATE  = "2020-01-01"   # Download from this date
WINDOW_SIZE = 60             # Days of history used to predict next day
TRAIN_RATIO = 0.70           # 70% train
VAL_RATIO   = 0.15           # 15% validation
# remaining 15% = test  (all splits are chronological — no shuffle)
EPOCHS      = 100
BATCH_SIZE  = 32

os.makedirs("models", exist_ok=True)
os.makedirs("plots",  exist_ok=True)

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.grid"]      = True

print(f"TensorFlow {tf.__version__}")

## 2. Download and Save Historical Data

`auto_adjust=True` returns prices that are already adjusted for splits and dividends.
If the download fails (e.g. no internet), the notebook falls back to the existing `stock_data.csv`.

In [ ]:
DATA_FILE = "stock_data.csv"

def download_data(ticker: str, start: str) -> pd.DataFrame:
    """Download OHLCV data from Yahoo Finance and return a clean DataFrame."""
    print(f"Downloading {ticker} from {start} …")
    raw = yf.Ticker(ticker).history(start=start, auto_adjust=True)

    if raw.empty:
        raise ValueError(f"yfinance returned an empty DataFrame for {ticker}.")

    # Flatten tz-aware DatetimeIndex to plain date strings
    raw.index = pd.to_datetime(raw.index).tz_localize(None)
    raw = raw.reset_index().rename(columns={"index": "Date"})

    # Keep only the columns we care about
    keep = ["Date", "Open", "High", "Low", "Close", "Volume"]
    raw = raw[[c for c in keep if c in raw.columns]].copy()

    # Clean
    raw["Date"]  = pd.to_datetime(raw["Date"], errors="coerce")
    raw["Close"] = pd.to_numeric(raw["Close"], errors="coerce")
    raw = raw.dropna(subset=["Date", "Close"])
    raw = raw[raw["Close"] > 0]
    raw = raw.drop_duplicates(subset=["Date"])
    raw = raw.sort_values("Date").reset_index(drop=True)

    # Format dates as YYYY-MM-DD
    raw["Date"] = raw["Date"].dt.strftime("%Y-%m-%d")
    return raw


try:
    df_new = download_data(TICKER, START_DATE)
    df_new.to_csv(DATA_FILE, index=False)
    print(f"Saved {len(df_new):,} rows → {DATA_FILE}")
    df = df_new.copy()
    df["Date"] = pd.to_datetime(df["Date"])
except Exception as e:
    print(f"Download failed: {e}")
    if os.path.exists(DATA_FILE):
        print(f"Falling back to existing {DATA_FILE}")
        df = pd.read_csv(DATA_FILE)
        df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
        df["Close"] = pd.to_numeric(df["Close"], errors="coerce")
        df = df.dropna(subset=["Date", "Close"]).sort_values("Date").reset_index(drop=True)
    else:
        raise RuntimeError("No data available. Please check your internet connection.")

print(f"\nDataset shape : {df.shape}")
print(f"Date range    : {df['Date'].min().date()} → {df['Date'].max().date()}")
print(f"Columns       : {df.columns.tolist()}")
print("\nNote: prices are ADJUSTED closes (splits + dividends removed via yfinance auto_adjust=True).")

## 3. Data Exploration

In [ ]:
print(df.describe())
print("\nMissing values:")
print(df.isnull().sum())

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df["Date"], df["Close"], label="Close (adjusted)", linewidth=1, color="#2563eb")
ax.plot(df["Date"], df["Close"].rolling(50).mean(),  label="MA 50",  linewidth=1.2, linestyle="--")
ax.plot(df["Date"], df["Close"].rolling(200).mean(), label="MA 200", linewidth=1.2, linestyle=":")
ax.set_title("RELIANCE.NS — Adjusted Closing Price (2020 – present)")
ax.set_xlabel("Date")
ax.set_ylabel("Price (₹)")
ax.legend()
plt.tight_layout()
plt.savefig("plots/price_history.png", dpi=120)
plt.show()

In [ ]:
# Daily returns
returns = df["Close"].pct_change().dropna() * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(returns, bins=80, color="#2563eb", edgecolor="white", linewidth=0.4)
axes[0].set_title("Daily Returns Distribution (%)")
axes[0].set_xlabel("Return (%)")
axes[1].plot(df["Date"].iloc[1:], returns.values, linewidth=0.6, color="#2563eb")
axes[1].set_title("Daily Returns Over Time (%)")
axes[1].set_xlabel("Date")
plt.tight_layout()
plt.show()

print(f"Mean daily return : {returns.mean():.3f}%")
print(f"Std daily return  : {returns.std():.3f}%")
print(f"Max daily return  : {returns.max():.3f}%")
print(f"Min daily return  : {returns.min():.3f}%")

## 4. Chronological Train / Validation / Test Split

All splits are **strictly chronological — no shuffling**.
- **Train (70%)** — used to fit the scaler and train the models.
- **Validation (15%)** — used by EarlyStopping to select the best epoch.
- **Test (15%)** — held-out; never seen during training or hyper-parameter selection.

The `MinMaxScaler` is fitted **only on training prices**. Applying it to the validation and test sets does not leak any information because the scaler does not re-estimate its parameters.

In [ ]:
prices = df["Close"].to_numpy().reshape(-1, 1)  # shape (N, 1)
N = len(prices)

train_end = int(N * TRAIN_RATIO)
val_end   = int(N * (TRAIN_RATIO + VAL_RATIO))
# test: val_end → N

print(f"Total trading days : {N}")
print(f"Train  → rows  0 … {train_end-1}  ({train_end} days)  "
      f"{df['Date'].iloc[0].date()} → {df['Date'].iloc[train_end-1].date()}")
print(f"Val    → rows {train_end} … {val_end-1}  ({val_end-train_end} days)  "
      f"{df['Date'].iloc[train_end].date()} → {df['Date'].iloc[val_end-1].date()}")
print(f"Test   → rows {val_end} … {N-1}  ({N-val_end} days)  "
      f"{df['Date'].iloc[val_end].date()} → {df['Date'].iloc[-1].date()}")

# Fit scaler on training data ONLY
scaler = MinMaxScaler()
scaler.fit(prices[:train_end])
scaled = scaler.transform(prices)  # transform the full series

print(f"\nScaler fitted on training prices: min={scaler.data_min_[0]:.2f}, max={scaler.data_max_[0]:.2f}")

## 5. Build Sliding-Window Sequences

Each sample: **60 consecutive scaled closes → next-day scaled close**.
Sequences are assigned to train, val or test based on the position of the *target* day.

In [ ]:
X_train, y_train = [], []
X_val,   y_val   = [], []
X_test,  y_test  = [], []

for i in range(WINDOW_SIZE, N):
    seq    = scaled[i - WINDOW_SIZE: i]          # shape (60, 1)
    target = scaled[i]                            # shape (1,)
    if i < train_end:
        X_train.append(seq); y_train.append(target)
    elif i < val_end:
        X_val.append(seq);   y_val.append(target)
    else:
        X_test.append(seq);  y_test.append(target)

X_train = np.array(X_train, dtype=np.float32)
y_train = np.array(y_train, dtype=np.float32)
X_val   = np.array(X_val,   dtype=np.float32)
y_val   = np.array(y_val,   dtype=np.float32)
X_test  = np.array(X_test,  dtype=np.float32)
y_test  = np.array(y_test,  dtype=np.float32)

print("X_train:", X_train.shape, "  y_train:", y_train.shape)
print("X_val  :", X_val.shape,   "  y_val  :", y_val.shape)
print("X_test :", X_test.shape,  "  y_test :", y_test.shape)

## 6. Define Models

In [ ]:
def build_mlp():
    return Sequential([
        Input(shape=(WINDOW_SIZE, 1)),
        Flatten(),
        Dense(64, activation="relu"),
        Dropout(0.2),
        Dense(32, activation="relu"),
        Dense(1),
    ], name="MLP")


def build_lstm():
    return Sequential([
        Input(shape=(WINDOW_SIZE, 1)),
        LSTM(50, return_sequences=True),
        LSTM(50),
        Dense(1),
    ], name="LSTM")


def build_gru():
    return Sequential([
        Input(shape=(WINDOW_SIZE, 1)),
        GRU(50, return_sequences=True),
        GRU(50),
        Dense(1),
    ], name="GRU")


arch = {"MLP": build_mlp, "LSTM": build_lstm, "GRU": build_gru}
for name, fn in arch.items():
    m = fn()
    print(f"{name}: {m.count_params():,} parameters")

## 7. Train All Three Models

In [ ]:
histories = {}
trained   = {}

early_stop = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)

for name, build_fn in arch.items():
    print(f"\n{'='*50}\nTRAINING {name}\n{'='*50}")
    model = build_fn()
    model.compile(optimizer="adam", loss="mean_squared_error")

    hist = model.fit(
        X_train, y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(X_val, y_val),   # use explicit val split, not fraction
        shuffle=False,                     # never shuffle a time series
        callbacks=[early_stop],
        verbose=1,
    )
    histories[name] = hist
    trained[name]   = model

print("\nAll models trained.")

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
for ax, (name, h) in zip(axes, histories.items()):
    n_epochs = len(h.history["loss"])
    ax.plot(h.history["loss"],     label="Train loss")
    ax.plot(h.history["val_loss"], label="Val loss")
    ax.set_title(f"{name} ({n_epochs} epochs)")
    ax.set_xlabel("Epoch")
    ax.legend()
axes[0].set_ylabel("MSE")
plt.tight_layout()
plt.savefig("plots/training_loss.png", dpi=120)
plt.show()

## 8. Evaluate on the Test Set

All models are evaluated on the **same held-out test period** using MAE and RMSE in the original price scale (₹).

The **persistence baseline** predicts tomorrow's close = today's close. Because stock prices are close to a random walk, the baseline is a tough benchmark.

In [ ]:
# Actual test prices in ₹
actual_test = scaler.inverse_transform(y_test).flatten()        # shape (n_test,)
prev_test   = prices[val_end - 1 : N - 1].flatten()            # yesterday's actual close
test_dates  = df["Date"].iloc[val_end:].to_numpy()              # dates for plotting

print(f"Test period: {pd.Timestamp(test_dates[0]).date()} → {pd.Timestamp(test_dates[-1]).date()}")
print(f"Test days  : {len(actual_test)}")

# Collect predictions
preds = {}
for name, model in trained.items():
    p = model.predict(X_test, verbose=0, batch_size=256)
    preds[name] = scaler.inverse_transform(p).flatten()
preds["Persistence Baseline"] = prev_test


def metrics(actual, pred):
    mae  = mean_absolute_error(actual, pred)
    rmse = np.sqrt(mean_squared_error(actual, pred))
    return mae, rmse


rows = []
for name, pred in preds.items():
    mae, rmse = metrics(actual_test, pred)
    rows.append({"Model": name, "MAE (₹)": mae, "RMSE (₹)": rmse})

results_df = pd.DataFrame(rows).sort_values("MAE (₹)").reset_index(drop=True)
print("\n" + results_df.round(2).to_string(index=False))

In [ ]:
# Actual vs predicted chart
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(test_dates, actual_test, label="Actual", color="black", linewidth=2)
colors = {"MLP": "#EF553B", "LSTM": "#00CC96", "GRU": "#AB63FA"}
for name, pred in preds.items():
    if name != "Persistence Baseline":
        ax.plot(test_dates, pred, label=name, linewidth=1.2,
                color=colors.get(name, "gray"))
ax.plot(test_dates, preds["Persistence Baseline"], label="Persistence Baseline",
        linewidth=1, linestyle="--", color="#FFA15A")
ax.set_title("Test Set: Actual vs Predicted Closing Prices")
ax.set_xlabel("Date")
ax.set_ylabel("Price (₹)")
ax.legend()
plt.tight_layout()
plt.savefig("plots/actual_vs_predicted.png", dpi=120)
plt.show()

print("\n⚠️  A model that produces a smooth, shifted version of the actual curve can still"
      "\n   have a high MAE/RMSE. The metrics table above is the honest comparison.")

In [ ]:
# Bar chart: MAE and RMSE
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
bar_colors = ["#FFA15A", "#EF553B", "#00CC96", "#AB63FA"]
for ax, metric in zip(axes, ["MAE (₹)", "RMSE (₹)"]):
    bars = ax.bar(results_df["Model"], results_df[metric], color=bar_colors[:len(results_df)])
    ax.bar_label(bars, fmt="₹%.1f", padding=3)
    ax.set_title(f"{metric} — lower is better")
    ax.tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.savefig("plots/metric_comparison.png", dpi=120)
plt.show()

## 9. Save Models and Scaler

In [ ]:
for name, model in trained.items():
    path = f"models/{name.lower()}_model.keras"
    model.save(path)
    print(f"Saved {path}")

joblib.dump(scaler, "models/price_scaler.pkl")
results_df.to_csv("model_results.csv", index=False)

# Save metadata so app.py knows the split boundaries
import json
meta = {
    "train_end_idx": int(train_end),
    "val_end_idx":   int(val_end),
    "window_size":   WINDOW_SIZE,
    "train_ratio":   TRAIN_RATIO,
    "val_ratio":     VAL_RATIO,
    "data_file":     "stock_data.csv",
    "price_type":    "adjusted_close (yfinance auto_adjust=True)",
    "trained_on":    str(pd.Timestamp.now().date()),
}
with open("models/meta.json", "w") as f:
    json.dump(meta, f, indent=2)

print("\nAll artifacts saved.")
print("Launch dashboard : streamlit run app.py")

## 10. Conclusions

- The **persistence baseline** (tomorrow = today) is a tough benchmark for daily price forecasting because stock prices behave close to a random walk.
- **GRU** tends to outperform MLP and LSTM on this task due to its gated recurrent structure with fewer parameters than LSTM.
- **Direction accuracy** around 50% means the models are essentially guessing movement direction — this is expected when training purely on closing prices.
- Only closing prices were used. Incorporating volume, technical indicators, or macroeconomic features could improve performance.
- For **multi-step / long-horizon forecasting**, recursive prediction error compounds: each step feeds the previous model prediction (not the true price) back into the window. Uncertainty grows with the horizon.

> **Educational project. Not financial advice.**

**Next step:** run `streamlit run app.py`